# Day counts and rates

Date conventions, discount factors and forward rates.


In [ ]:
import calendar
from datetime import date

import pandas as pd


## Month end


In [ ]:
def iseom(day):
    return day.day == calendar.monthrange(day.year, day.month)[1]


In [ ]:
iseom(date(2026, 2, 28))


## Day counts

All counts are positive or zero. `30I/360` adjusts month ends, except a February maturity date. The basic `30/360` calculation has no month-end adjustments.


In [ ]:
def count_days(date_1, date_2, method="30I/360", T=None):
    if method == "actual":
        return abs((date_2 - date_1).days)
    if method not in ("30I/360", "30/360"):
        raise ValueError("Choose actual, 30I/360 or 30/360.")

    d1, d2 = date_1.day, date_2.day
    if method == "30I/360":
        if iseom(date_1):
            d1 = 30
        if iseom(date_2) and (date_2 != T or date_2.month != 2):
            d2 = 30

    years = date_2.year - date_1.year
    months = date_2.month - date_1.month
    return abs(360 * years + 30 * months + d2 - d1)


In [ ]:
start = date(2026, 1, 31)
end = date(2026, 7, 31)
pd.Series({method: count_days(start, end, method=method)
           for method in ["actual", "30I/360", "30/360"]})


## Discount factors

`zero_rate=5` means 5%. For decimal rates, set `make_pct=False`. The default is semiannual compounding on a 360-day year.


In [ ]:
def discount_from_zero(date_1, date_2, zero_rate, make_pct=True, met='30I/360', freq=2):
    if make_pct:
        zero_rate /= 100
    num_days = count_days(date_1, date_2, method=met, T=date_2)
    return 1 / (1 + zero_rate / freq) ** (freq * num_days / 360)


In [ ]:
discount_from_zero(date(2026, 1, 1), date(2027, 1, 1), zero_rate=5.0)


## Forward rates

Discount factors are indexed by date, in increasing order. Forward rates are in percent, using actual days / 360.


In [ ]:
def forward_from_discount(discounts):
    if len(discounts) < 2:
        return pd.Series(dtype=float)

    forwards = []
    for i in range(len(discounts) - 1):
        start, end = discounts.index[i:i + 2]
        days = count_days(start, end, method="actual")
        ratio = discounts.iloc[i] / discounts.iloc[i + 1]
        forwards.append((ratio - 1) * 360 / days * 100)
    return pd.Series(forwards, index=discounts.index[:-1], dtype=float)


In [ ]:
discounts = pd.Series(
    [1.0, 0.975, 0.95],
    index=[date(2026, 1, 1), date(2026, 7, 1), date(2027, 1, 1)],
)
forward_from_discount(discounts)
